In [ ]:
import os

from openai import max_retries

os.environ.setdefault("USER_AGENT", "research-agent/1.0")

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=3000)
)
duckduckgo = DuckDuckGoSearchResults(output_format="list", max_results=5)

@tool
def wikipedia_search(query: str) -> str:
    """위키피디아에서 주제를 검색해 관련 문서의 요약을 돌려준다. 개념이나 사건의 배경을 알아볼 때 쓴다."""
    try:
        return wikipedia.invoke(query)
    except Exception as e:
        return f"위키피디아 검색에 실패했습니다: {e}"

@tool
def duckduckgo_search(query: str) -> str:
    """DuckDuckGo에서 웹을 검색해 결과마다 제목, 링크(URL), 요약을 돌려준다. 자세히 읽을 웹사이트를 찾을 때 쓴다."""
    try:
        results = duckduckgo.invoke(query)
    except Exception as e:
        return f"DuckDuckGo 검색에 실패했습니다: {e}"
    return "\n\n".join(
        f"제목: {r['title']}\n랑크: {r['link']}\n요약: {r['snippet']}" for r in results
    )

@tool
def scrape_website(url: str) -> str:
    """웹사이트 URL을 받아 페이지의 본문 텍스트를 추출한다. DuckDuckGo에서 찾은 링크의 내용을 읽을 때 쓴다."""
    try:
        soup = WebBaseLoader(url).scrape()
    except Exception as e:
        return f"웹세이트를 읽지 못했어요: {e}"
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()
    body = soup.find("main") or soup.find("article") or soup.body or soup
    text = " ".join(body.get_text(" ").split())
    return text[:5000]

@tool
def save_research(filename: str, content: str) -> str:
    """조사한 내용을 .txt 파일로 저장하고, 저장한 파일 경로를 돌려준다. 조사를 모두 마친 뒤 마지막에 쓴다."""
    os.makedirs("research", exist_ok=True)
    name = os.path.basename(filename)
    if not name.endswith(".txt"):
        name += ".txt"
    path = os.path.join("research", name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"{path}에 저장했어요. ({len(content)}자)"

tools = [wikipedia_search, duckduckgo_search, scrape_website, save_research]

In [ ]:
llm = ChatOpenAI(model="gpt-5.6-terra", use_responses_api=True)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""당신은 꼼꼼한 리서치 에이전트입니다. 다음 순서로 조사하세요.
    1. wikipedia_search와 duckduckgo_search로 주제를 검색합니다. 둘 중 하나가 실패하면 다른 하나로 계속하세요
    2. duckduckgo_search 결과에서 믿을 만한 웹사이트 1~2개를 골라 scrape_website로 내용을 읽습니다
    3. 모든 내용을 한국어로 정리합니다. 개요, 주요 사실(날짜, 버전, 관련 인물등), 영향, 출처 URL 목록을 넣으세요
    4. 마지막으로 save_research로 정리한 내용을 .txt 파일에 저장하고, 저장한 파일 경로를 알려 주세요""",
)

In [ ]:
def short(value, limit=150):
    text = str(value).replace("\n", " ")
    return text if len(text) <= limit else text[:limit] + "..."

query = "Research about the XZ backdoor"
for step in agent.stream({"messages": [{"role": "user", "content": query}]}, stream_mode="updates"):
    for update in step.values():
        for message in update.get("messages", []):
            if message.type == "ai" and message.tool_calls:
                for call in message.tool_calls:
                    args = {k: short(v, 80) for k, v in call["args"].items()}
                    print(f"{call['name']} 호출 -> {args}")
            elif message.type == "tool":
                print(f"{message.name} 결과 -> {short(message.content)}\n")
            elif message.type == "ai":
                print("\n최종 답변\n{message.text")

In [ ]:
import glob

for path in glob.glob("research/*.txt"):
    print(f"===== {path} =====")
    with open(path, encoding="utf-8") as f:
        print(f.read())